In [1]:
import requests
import pandas as pd
import defs

In [2]:
session = requests.Session()

In [3]:
instr_df = pd.read_pickle('instruments.pkl')

In [4]:
our_currencies = ['EUR', 'USD', 'JPY', 'GBP', 'AUD', 'CAD']

In [19]:
# Function that fetches candlestick data for a given instrument

def fetch_candles(pair_name, count, granularity):
    url = f"{defs.OANDA_URL}/instruments/{pair_name}/candles"
    params = {
        'count': count,
        'granularity': granularity,
        'price': 'MBA'
    }
    response = session.get(url, params=params, headers=defs.SECURE_HEADER)
    return response.status_code, response.json()

In [20]:
code, res = fetch_candles('EUR_USD', 10, 'H1')


In [22]:
# Function that takes the response from the API and creates a DataFrame

def get_candles_df(response_json):
    prices = ['mid', 'bid', 'ask']
    ohlc = ['o', 'h', 'l', 'c']
    
    our_data = []
    for candle in response_json['candles']:
        if not candle['complete']:
            continue
        new_dict = {}
        new_dict['time'] = candle['time']
        new_dict['volume'] = candle['volume']
        for price in prices: 
            for o in ohlc:
                new_dict[f'{price}_{o}'] = candle[price][o]
        our_data.append(new_dict)

    return pd.DataFrame(our_data)


In [13]:
# Function that saves the DataFrame to a PKL file

def save_file(candles_df, pair, granularity):
    candles_df.to_pickle(f'hist_data/{pair}_{granularity}.pkl')

In [14]:
# Function that creates the data we need to save into the fles
# Controls the flow of execution

def create_data(pair, granularity):
    code, json_data = fetch_candles(pair, 4000, granularity)
    if code != 200:
        print(pair, 'Error')
        return
    
    df = get_candles_df(json_data)
    print(f"{pair} loaded {df.shape[0]} candles from {df.time.min()} to {df.time.max()}")
    save_file(df, pair, granularity)

In [23]:
for p1 in our_currencies:
    for p2 in our_currencies:
        pair = f"{p1}_{p2}"
        if pair in instr_df.name.unique():
            create_data(pair, 'H1')


EUR_USD loaded 3999 candles from 2025-09-25T22:00:00.000000000Z to 2026-05-20T12:00:00.000000000Z
EUR_JPY loaded 3999 candles from 2025-09-25T22:00:00.000000000Z to 2026-05-20T12:00:00.000000000Z
EUR_GBP loaded 3999 candles from 2025-09-25T22:00:00.000000000Z to 2026-05-20T12:00:00.000000000Z
EUR_AUD loaded 3999 candles from 2025-09-25T22:00:00.000000000Z to 2026-05-20T12:00:00.000000000Z
EUR_CAD loaded 3999 candles from 2025-09-25T22:00:00.000000000Z to 2026-05-20T12:00:00.000000000Z
USD_JPY loaded 3999 candles from 2025-09-25T22:00:00.000000000Z to 2026-05-20T12:00:00.000000000Z
USD_CAD loaded 3999 candles from 2025-09-25T22:00:00.000000000Z to 2026-05-20T12:00:00.000000000Z
GBP_USD loaded 3999 candles from 2025-09-25T22:00:00.000000000Z to 2026-05-20T12:00:00.000000000Z
GBP_JPY loaded 3999 candles from 2025-09-25T22:00:00.000000000Z to 2026-05-20T12:00:00.000000000Z
GBP_AUD loaded 3999 candles from 2025-09-25T22:00:00.000000000Z to 2026-05-20T12:00:00.000000000Z
GBP_CAD loaded 3999 